In [8]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, f1_score

train_df = pd.read_csv(
    "../../data/processed/selected_training.csv"
)

test_df = pd.read_csv(
    "../../data/processed/selected_testing.csv"
)


In [5]:
from sklearn.model_selection import train_test_split

X = train_df.drop(columns=["label", "attack_cat"])

y = train_df["attack_cat"].fillna("Normal")
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)

Training: (98869, 43)
Validation: (24718, 43)


In [15]:
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical:", numeric_features)
print("Categorical:", categorical_features)

Numerical: ['id', 'dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports']
Categorical: ['proto', 'service', 'state']


In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# Even though you currently have no missing values, the imputers make the pipeline more robust. 

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Unknown"
    )),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

In [17]:
print("Preprocessor:", preprocessor)
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

Preprocessor: ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['id', 'dur', 'spkts', 'dpkts', 'sbytes',
                                  'dbytes', 'rate', 'sttl', 'dttl', 'sload',
                                  'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt',
                                  'sjit', 'djit', 'swin', 'stcpb', 'dtcpb',
                                  'dwin', 'tcprtt', 'synack', 'ackdat', 'smean',
                                  'dmean', 'trans_depth', 'response_body_len',
                                  'ct_srv_src', 'ct_state_ttl', ...]),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Unknown',
                                                           

## Training the baseline model 

In [22]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, f1_score

# Create baseline model
baseline_model = DummyClassifier(
    strategy="most_frequent"
)

# Train the baseline
baseline_model.fit(X_train, y_train)

# Predict on validation data
y_pred_base = baseline_model.predict(X_val)

# Evaluate
print(
    "Baseline Macro F1:",
    f1_score(y_val, y_pred_base, average="macro")
)

print(
    classification_report(
        y_val,
        y_pred_base,
        zero_division=0
    )
)

Baseline Macro F1: 0.10394045696679473
              precision    recall  f1-score   support

    Analysis       0.00      0.00      0.00       400
    Backdoor       0.00      0.00      0.00       349
         DoS       0.00      0.00      0.00      2453
    Exploits       0.00      0.00      0.00      6679
     Fuzzers       0.00      0.00      0.00      3637
      Normal       0.45      1.00      0.62     11200

    accuracy                           0.45     24718
   macro avg       0.08      0.17      0.10     24718
weighted avg       0.21      0.45      0.28     24718



## Training the actual ML model

In [18]:
from sklearn.tree import DecisionTreeClassifier

decision_tree = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        random_state=42
    ))
])

decision_tree.fit(X_train, y_train)

y_pred_dt = decision_tree.predict(X_val)

print("Decision Tree Macro F1:",
      f1_score(y_val, y_pred_dt, average="macro"))

print(classification_report(y_val, y_pred_dt, zero_division=0))

Decision Tree Macro F1: 0.5749809688979841
              precision    recall  f1-score   support

    Analysis       0.23      0.23      0.23       400
    Backdoor       0.23      0.19      0.21       349
         DoS       0.38      0.41      0.40      2453
    Exploits       0.77      0.76      0.76      6679
     Fuzzers       0.87      0.85      0.86      3637
      Normal       0.99      0.99      0.99     11200

    accuracy                           0.83     24718
   macro avg       0.58      0.57      0.57     24718
weighted avg       0.83      0.83      0.83     24718



## Train Random Forest

In [19]:
from sklearn.ensemble import RandomForestClassifier

random_forest = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

random_forest.fit(X_train, y_train)

y_pred_rf = random_forest.predict(X_val)

print("Random Forest Macro F1:",
      f1_score(y_val, y_pred_rf, average="macro"))

print(classification_report(y_val, y_pred_rf, zero_division=0))

Random Forest Macro F1: 0.5639166496539884
              precision    recall  f1-score   support

    Analysis       0.23      0.20      0.21       400
    Backdoor       0.20      0.15      0.17       349
         DoS       0.40      0.37      0.39      2453
    Exploits       0.77      0.81      0.79      6679
     Fuzzers       0.82      0.87      0.85      3637
      Normal       0.99      0.97      0.98     11200

    accuracy                           0.83     24718
   macro avg       0.57      0.56      0.56     24718
weighted avg       0.82      0.83      0.83     24718



In [20]:
from sklearn.ensemble import RandomForestClassifier

random_forest = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

random_forest.fit(X_train, y_train)

y_pred_rf = random_forest.predict(X_val)

print("Random Forest Macro F1:",
      f1_score(y_val, y_pred_rf, average="macro"))

print(classification_report(
    y_val,
    y_pred_rf,
    zero_division=0
))

Random Forest Macro F1: 0.5639166496539884
              precision    recall  f1-score   support

    Analysis       0.23      0.20      0.21       400
    Backdoor       0.20      0.15      0.17       349
         DoS       0.40      0.37      0.39      2453
    Exploits       0.77      0.81      0.79      6679
     Fuzzers       0.82      0.87      0.85      3637
      Normal       0.99      0.97      0.98     11200

    accuracy                           0.83     24718
   macro avg       0.57      0.56      0.56     24718
weighted avg       0.82      0.83      0.83     24718



In [23]:
results = pd.DataFrame({
    "Model": [
        "Dummy Classifier",
        "Decision Tree",
        "Random Forest"
    ],
    "Macro F1": [
        f1_score(y_val, y_pred_base, average="macro"),
        f1_score(y_val, y_pred_dt, average="macro"),
        f1_score(y_val, y_pred_rf, average="macro")
    ]
})

print(results.sort_values(
    by="Macro F1",
    ascending=False
))

              Model  Macro F1
1     Decision Tree  0.574981
2     Random Forest  0.563917
0  Dummy Classifier  0.103940
